In [1]:
import pandas as pd
import rasterio
from rasterio.merge import merge
from rasterio.warp import transform
import pyproj
import numpy as np
import glob
import os
import sys

In [2]:
DGM_DIR = r'C:\Users\phili\Documents\WS2526\Measurement project\Project\Skript\qgis\DGM1 GIeßen'
GPS_FILE = r'Location.csv' # Muss im aktuellen Arbeitsverzeichnis liegen
OUTPUT_FILE = r'GPS_DGM.csv'

CRS_WGS84 = 'epsg:4326' # WGS84 Lat/Lon
CRS_DGM = 'epsg:25832' # ETRS89 / UTM 32N

In [3]:
print("Schritt 1: Lade GPS-Daten...")
try:
    gps_data = pd.read_csv(GPS_FILE)
    gps_data.drop_duplicates(subset=['Time (s)'], keep='first', inplace=True)
    gps_data.sort_values(by='Time (s)', inplace=True)
    gps_data.reset_index(drop=True, inplace=True)
    latitudes = gps_data['Latitude (°)'].values
    longitudes = gps_data['Longitude (°)'].values
except FileNotFoundError:
    print(f"FEHLER: {GPS_FILE} nicht gefunden.")
    sys.exit(1)

Schritt 1: Lade GPS-Daten...


In [4]:
print("Schritt 2: Lade DGM-Kacheln und erstelle Mosaik...")

src_files_to_mosaic = []
for fp in glob.glob(os.path.join(DGM_DIR, '*.tif')):
    try:
        src = rasterio.open(fp)
        src_files_to_mosaic.append(src)
    except rasterio.RasterioIOError:
        print(f"WARNUNG: Kann Kachel {os.path.basename(fp)} nicht öffnen.")

if not src_files_to_mosaic:
    print(f"FEHLER: Keine DGM-Kacheln gefunden.")
    sys.exit(1)

print(f" > {len(src_files_to_mosaic)} Kacheln geladen. Erstelle Mosaik...")
mosaic, out_trans = merge(src_files_to_mosaic)
mosaic_nodata = src_files_to_mosaic[0].nodata # Geht davon aus, dass alle Kacheln gleiche NoData haben

#Alle geöffneten Raster-Dateien schließen

for src in src_files_to_mosaic:
    src.close()


Schritt 2: Lade DGM-Kacheln und erstelle Mosaik...
 > 109 Kacheln geladen. Erstelle Mosaik...


In [5]:
print("Schritt 3: Transformiere GPS-Punkte in DGM-CRS (UTM 32N)...")
transformer = pyproj.Transformer.from_crs(CRS_WGS84, CRS_DGM, always_xy=True)
easting, northing = transformer.transform(longitudes, latitudes)

Schritt 3: Transformiere GPS-Punkte in DGM-CRS (UTM 32N)...


In [6]:
print("Schritt 4: Extrahiere DGM-Höhen für jeden GPS-Punkt...")
dgm_altitudes = np.full(len(easting), np.nan)

for i, (e, n) in enumerate(zip(easting, northing)):
    # Pixel-Koordinaten im Mosaik
    row, col = ~out_trans * (e, n)
    row, col = int(row), int(col)
    # Prüfe, ob innerhalb des Mosaiks
    if 0 <= row < mosaic.shape[1] and 0 <= col < mosaic.shape[2]:
        value = mosaic[0, row, col] # Band 1
        # Prüfe auf NoData
    if mosaic_nodata is None or value != mosaic_nodata:
        dgm_altitudes[i] = value
    else:
        print(f" > Punkt {i} ({e:.2f}, {n:.2f}) liegt außerhalb des Mosaiks.")

Schritt 4: Extrahiere DGM-Höhen für jeden GPS-Punkt...


In [7]:
print("Schritt 5: Speichere Ergebnisse...")
gps_data['DGM_Height_m'] = dgm_altitudes
gps_data.to_csv(OUTPUT_FILE, index=False)
print(f"✅ Fertig. Augmented CSV gespeichert: {OUTPUT_FILE}")

Schritt 5: Speichere Ergebnisse...
✅ Fertig. Augmented CSV gespeichert: GPS_DGM.csv


In [8]:
print(f"{len(src_files_to_mosaic)} Kacheln gefunden")
for src in src_files_to_mosaic:
    print(src.name)


109 Kacheln gefunden
C:\Users\phili\Documents\WS2526\Measurement project\Project\Skript\qgis\DGM1 GIeßen\dgm1_32_468_5597_1_he.tif
C:\Users\phili\Documents\WS2526\Measurement project\Project\Skript\qgis\DGM1 GIeßen\dgm1_32_468_5598_1_he.tif
C:\Users\phili\Documents\WS2526\Measurement project\Project\Skript\qgis\DGM1 GIeßen\dgm1_32_469_5597_1_he.tif
C:\Users\phili\Documents\WS2526\Measurement project\Project\Skript\qgis\DGM1 GIeßen\dgm1_32_469_5598_1_he.tif
C:\Users\phili\Documents\WS2526\Measurement project\Project\Skript\qgis\DGM1 GIeßen\dgm1_32_469_5599_1_he.tif
C:\Users\phili\Documents\WS2526\Measurement project\Project\Skript\qgis\DGM1 GIeßen\dgm1_32_470_5597_1_he.tif
C:\Users\phili\Documents\WS2526\Measurement project\Project\Skript\qgis\DGM1 GIeßen\dgm1_32_470_5598_1_he.tif
C:\Users\phili\Documents\WS2526\Measurement project\Project\Skript\qgis\DGM1 GIeßen\dgm1_32_470_5599_1_he.tif
C:\Users\phili\Documents\WS2526\Measurement project\Project\Skript\qgis\DGM1 GIeßen\dgm1_32_471_559